In [1]:
# film_tests.py
# Minimal FiLM-conditioning test framework for geoscience-style toy problems
# Requirements: python>=3.10, torch, numpy, matplotlib

from __future__ import annotations
from dataclasses import dataclass
from typing import Literal, Tuple, Dict, Optional
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt


# ----------------------------
# Utilities
# ----------------------------

def set_seed(seed: int = 0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def to_torch(x: np.ndarray, device: torch.device) -> torch.Tensor:
    return torch.tensor(x, dtype=torch.float32, device=device)

def periodic_roll(x: torch.Tensor, shift: int, dim: int = -1) -> torch.Tensor:
    return torch.roll(x, shifts=shift, dims=dim)

def grad_x_periodic(u: torch.Tensor, dx: float) -> torch.Tensor:
    # central difference, periodic
    return (periodic_roll(u, -1) - periodic_roll(u, 1)) / (2.0 * dx)

def laplace_x_periodic(u: torch.Tensor, dx: float) -> torch.Tensor:
    return (periodic_roll(u, -1) - 2.0*u + periodic_roll(u, 1)) / (dx*dx)

def mass(u: torch.Tensor) -> torch.Tensor:
    return u.mean(dim=-1)

def energy(u: torch.Tensor) -> torch.Tensor:
    return (u*u).mean(dim=-1)

def plot_series(y: np.ndarray, title: str, out_png: str):
    plt.figure()
    plt.plot(y)
    plt.title(title)
    plt.xlabel("step")
    plt.ylabel("value")
    plt.tight_layout()
    plt.savefig(out_png, dpi=150)
    plt.close()


# ----------------------------
# Conditioning methods
# ----------------------------

class FiLM1d(nn.Module):
    """
    Feature-wise linear modulation for 1D feature maps: (B, C, X)
    Produces gamma, beta from conditioning vector c: (B, cond_dim)
    """
    def __init__(self, cond_dim: int, channels: int, hidden: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(cond_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, 2*channels),
        )

    def forward(self, h: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        gb = self.mlp(c)                           # (B, 2C)
        gamma, beta = gb.chunk(2, dim=-1)          # (B, C), (B, C)
        gamma = gamma.unsqueeze(-1)                # (B, C, 1)
        beta  = beta.unsqueeze(-1)                 # (B, C, 1)
        return gamma * h + beta


# ----------------------------
# Simple backbone (1D CNN)
# ----------------------------

class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, k: int = 5):
        super().__init__()
        pad = k // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size=k, padding=pad)
        self.act = nn.SiLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.conv(x))

class ToyNet(nn.Module):
    """
    Predicts next state u_{t+1} from current state u_t (and conditioning).
    Modes:
      - none: no conditioning used
      - concat: concat conditioning (broadcast) to input channels
      - film: FiLM modulation in hidden blocks
    """
    def __init__(
        self,
        x_channels: int,
        cond_dim: int,
        mode: Literal["none", "concat", "film"] = "film",
        width: int = 64,
        depth: int = 4,
    ):
        super().__init__()
        self.mode = mode
        self.cond_dim = cond_dim

        in_ch = x_channels
        if mode == "concat":
            in_ch = x_channels + cond_dim  # broadcast cond as extra channels

        self.in_proj = ConvBlock(in_ch, width)
        self.blocks = nn.ModuleList([ConvBlock(width, width) for _ in range(depth-2)])
        self.out_proj = nn.Conv1d(width, x_channels, kernel_size=1)

        if mode == "film":
            self.films = nn.ModuleList([FiLM1d(cond_dim, width) for _ in range(depth-1)])
        else:
            self.films = None

    def forward(self, x: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        # x: (B, Cx, X), c: (B, cond_dim)
        if self.mode == "concat":
            B, _, X = x.shape
            c_map = c.unsqueeze(-1).repeat(1, 1, X)   # (B, cond_dim, X)
            x = torch.cat([x, c_map], dim=1)

        h = self.in_proj(x)
        if self.mode == "film":
            h = self.films[0](h, c)

        for i, blk in enumerate(self.blocks, start=1):
            h = blk(h)
            if self.mode == "film":
                h = self.films[i](h, c)

        y = self.out_proj(h)
        return y


# ----------------------------
# Problem definitions
# ----------------------------

@dataclass
class Grid1D:
    nx: int = 128
    L: float = 2.0 * math.pi

    @property
    def dx(self) -> float:
        return self.L / self.nx

    def x(self) -> np.ndarray:
        return np.linspace(0.0, self.L, self.nx, endpoint=False)


@dataclass
class TrainCfg:
    device: str = "cpu"
    lr: float = 2e-3
    batch_size: int = 64
    steps: int = 2000
    print_every: int = 200
    rollout_steps: int = 200
    weight_decay: float = 0.0


# --- Test A: regime-switch advection-diffusion ---

@dataclass
class RegimePDECfg:
    dt: float = 0.01
    c_adv: float = 1.0               # advection speed
    kappa0: float = 0.01
    kappa1: float = 0.10
    noise_std: float = 0.00          # optional observation noise
    init_modes: Tuple[int, ...] = (1, 2, 3, 4)

def step_advdiff(u: torch.Tensor, grid: Grid1D, dt: float, c_adv: float, kappa: float) -> torch.Tensor:
    dx = grid.dx
    du_dx = grad_x_periodic(u, dx)
    d2u_dx2 = laplace_x_periodic(u, dx)
    return u + dt * (-c_adv * du_dx + kappa * d2u_dx2)

def sample_init_u(grid: Grid1D, batch: int, modes=(1,2,3,4), device=torch.device("cpu")) -> torch.Tensor:
    x = to_torch(grid.x()[None, :].repeat(batch, axis=0), device=device)  # (B, X)
    # random Fourier mixture
    u = torch.zeros_like(x)
    for m in modes:
        amp = (0.5 * torch.rand(batch, device=device) + 0.25)  # (B,)
        phs = 2*math.pi*torch.rand(batch, device=device)
        u = u + amp[:, None]*torch.sin(m*x + phs[:, None])
    # normalize amplitude
    u = u / (u.std(dim=-1, keepdim=True) + 1e-6)
    return u  # (B, X)

def make_batch_regime_pde(
    grid: Grid1D,
    cfg: RegimePDECfg,
    batch: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # regime r in {0,1}, conditioning vector c = [r] (float)
    r = torch.randint(0, 2, (batch,), device=device)
    kappa = torch.where(r == 0, torch.tensor(cfg.kappa0, device=device), torch.tensor(cfg.kappa1, device=device))
    u0 = sample_init_u(grid, batch=batch, modes=cfg.init_modes, device=device)          # (B, X)
    u1 = torch.stack([step_advdiff(u0[i], grid, cfg.dt, cfg.c_adv, float(kappa[i].item())) for i in range(batch)], dim=0)
    if cfg.noise_std > 0:
        u1 = u1 + cfg.noise_std * torch.randn_like(u1)
    x = u0.unsqueeze(1)  # (B, 1, X)
    y = u1.unsqueeze(1)  # (B, 1, X)
    c = r.float().unsqueeze(1)  # (B, 1)
    return x, c, y


# --- Test B: month/season linear toy ---

@dataclass
class SeasonalToyCfg:
    months: int = 12
    noise_std: float = 0.02

    # month-dependent amplitude & bias
    amp_min: float = 0.7
    amp_max: float = 1.5
    bias_min: float = -0.3
    bias_max: float = 0.3

def make_batch_seasonal_toy(
    grid: Grid1D,
    cfg: SeasonalToyCfg,
    batch: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    # month m in [0..11], conditioning as one-hot (B, 12)
    m = torch.randint(0, cfg.months, (batch,), device=device)
    c = F.one_hot(m, num_classes=cfg.months).float()  # (B, 12)

    # create input x field
    u = sample_init_u(grid, batch=batch, modes=(1,2,3,5), device=device)  # (B, X)

    # month-dependent affine transform
    # smooth seasonal variation
    phi = 2*math.pi*(m.float() / cfg.months)
    amp = cfg.amp_min + (cfg.amp_max - cfg.amp_min) * (0.5*(1.0 + torch.sin(phi)))
    bias = cfg.bias_min + (cfg.bias_max - cfg.bias_min) * (0.5*(1.0 + torch.cos(phi)))

    y = amp[:, None]*u + bias[:, None] + cfg.noise_std*torch.randn_like(u)

    x = u.unsqueeze(1)
    y = y.unsqueeze(1)
    return x, c, y, m


# --- Test C: resolution-conditioned closure (tau = nu(Δ) * du/dx) ---

@dataclass
class ClosureCfg:
    # treat "resolution" as dx scale. We'll sample a discrete set.
    dx_choices: Tuple[float, ...] = (0.5, 1.0, 2.0, 4.0)  # relative scale factors
    nu0: float = 0.10
    alpha: float = 0.7           # nu(dx) = nu0 * (dx)^alpha
    noise_std: float = 0.01

def make_batch_closure(
    grid: Grid1D,
    cfg: ClosureCfg,
    batch: int,
    device: torch.device
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    # pick dx scale factor s in choices; conditioning vector c = [log(s)]
    idx = torch.randint(0, len(cfg.dx_choices), (batch,), device=device)
    s = torch.tensor([cfg.dx_choices[i] for i in idx.cpu().numpy()], device=device, dtype=torch.float32)
    c = torch.log(s).unsqueeze(1)  # (B, 1)

    u = sample_init_u(grid, batch=batch, modes=(1,2,3,4,6), device=device)  # (B, X)
    # "effective dx" changes gradient magnitude; use scaled dx in derivative
    dx_eff = grid.dx * s  # (B,)
    # compute du/dx per-sample (loop is fine for toy)
    dudx = torch.stack([grad_x_periodic(u[i], float(dx_eff[i].item())) for i in range(batch)], dim=0)

    nu = cfg.nu0 * (s ** cfg.alpha)
    tau = nu[:, None] * dudx + cfg.noise_std * torch.randn_like(dudx)

    x = u.unsqueeze(1)
    y = tau.unsqueeze(1)
    return x, c, y, s


# ----------------------------
# Training / Evaluation
# ----------------------------

def train_model(
    *,
    problem: Literal["pde", "season", "closure"],
    mode: Literal["none", "concat", "film"],
    train_cfg: TrainCfg,
    grid: Grid1D,
    pde_cfg: RegimePDECfg | None = None,
    season_cfg: SeasonalToyCfg | None = None,
    closure_cfg: ClosureCfg | None = None,
    holdout: Optional[Dict] = None,
    out_prefix: str = "run",
):
    device = torch.device(train_cfg.device)
    if problem == "pde":
        cond_dim = 1
        assert pde_cfg is not None
    elif problem == "season":
        assert season_cfg is not None
        cond_dim = season_cfg.months
    elif problem == "closure":
        cond_dim = 1
        assert closure_cfg is not None
    else:
        raise ValueError(problem)

    model = ToyNet(x_channels=1, cond_dim=cond_dim, mode=mode).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=train_cfg.lr, weight_decay=train_cfg.weight_decay)

    def sample_batch():
        if problem == "pde":
            x, c, y = make_batch_regime_pde(grid, pde_cfg, train_cfg.batch_size, device)
            # holdout option: exclude regime 1 during training, etc.
            if holdout and holdout.get("exclude_regime") is not None:
                ex = int(holdout["exclude_regime"])
                mask = (c[:, 0] != float(ex))
                if mask.sum() < 4:  # fallback
                    return sample_batch()
                x, c, y = x[mask], c[mask], y[mask]
            return x, c, y

        if problem == "season":
            x, c, y, m = make_batch_seasonal_toy(grid, season_cfg, train_cfg.batch_size, device)
            if holdout and holdout.get("exclude_months") is not None:
                excl = set(int(mm) for mm in holdout["exclude_months"])
                mask = torch.tensor([int(mi.item()) not in excl for mi in m], device=device, dtype=torch.bool)
                if mask.sum() < 4:
                    return sample_batch()
                x, c, y = x[mask], c[mask], y[mask]
            return x, c, y

        if problem == "closure":
            x, c, y, s = make_batch_closure(grid, closure_cfg, train_cfg.batch_size, device)
            if holdout and holdout.get("exclude_scales") is not None:
                excl = set(float(ss) for ss in holdout["exclude_scales"])
                mask = torch.tensor([float(si.item()) not in excl for si in s], device=device, dtype=torch.bool)
                if mask.sum() < 4:
                    return sample_batch()
                x, c, y = x[mask], c[mask], y[mask]
            return x, c, y

        raise RuntimeError

    model.train()
    for step in range(1, train_cfg.steps + 1):
        x, c, y = sample_batch()
        pred = model(x, c if mode != "none" else torch.zeros_like(c))
        loss = F.mse_loss(pred, y)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % train_cfg.print_every == 0:
            print(f"[{problem}|{mode}] step {step:5d}/{train_cfg.steps}  loss={loss.item():.4e}")

    # --- save FiLM gamma/beta diagnostics (if applicable) ---
    if mode == "film":
        # probe conditioning space with a small grid
        model.eval()
        with torch.no_grad():
            if problem == "pde":
                C = torch.tensor([[0.0],[1.0]], device=device)
                labels = ["r=0", "r=1"]
            elif problem == "season":
                C = torch.eye(season_cfg.months, device=device)
                labels = [f"m{m+1}" for m in range(season_cfg.months)]
            else:  # closure
                scales = np.array(closure_cfg.dx_choices, dtype=float)
                C = torch.tensor(np.log(scales)[:, None], device=device, dtype=torch.float32)
                labels = [f"s={s:g}" for s in scales]

            # access first FiLM module only (enough to see behavior)
            film0: FiLM1d = model.films[0]  # type: ignore
            gb = film0.mlp(C).cpu().numpy()  # (Nc, 2*width)
            width = gb.shape[1] // 2
            gamma = gb[:, :width].mean(axis=1)  # average over channels
            beta  = gb[:, width:].mean(axis=1)

            plt.figure()
            plt.plot(gamma, marker="o")
            plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
            plt.title(f"Avg gamma vs condition ({problem})")
            plt.tight_layout()
            plt.savefig(f"{out_prefix}_{problem}_{mode}_gamma.png", dpi=150)
            plt.close()

            plt.figure()
            plt.plot(beta, marker="o")
            plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
            plt.title(f"Avg beta vs condition ({problem})")
            plt.tight_layout()
            plt.savefig(f"{out_prefix}_{problem}_{mode}_beta.png", dpi=150)
            plt.close()

    return model


@torch.no_grad()
def rollout_pde(
    model: nn.Module,
    grid: Grid1D,
    pde_cfg: RegimePDECfg,
    *,
    mode: Literal["none", "concat", "film"],
    r: int,
    steps: int,
    device: torch.device,
) -> Dict[str, np.ndarray]:
    model.eval()
    u = sample_init_u(grid, batch=1, device=device).unsqueeze(1)  # (1,1,X)
    c = torch.tensor([[float(r)]], device=device)

    masses = []
    energies = []
    errs = []

    # generate a "truth" rollout using the actual PDE
    u_truth = u.clone()
    for _ in range(steps):
        u_truth = step_advdiff(u_truth[0,0], grid, pde_cfg.dt, pde_cfg.c_adv, pde_cfg.kappa0 if r == 0 else pde_cfg.kappa1).view(1,1,-1)

    # now reset and rollout the learned model
    u = u.clone()
    for _ in range(steps):
        pred = model(u, c if mode != "none" else torch.zeros_like(c))
        u = pred

        masses.append(float(mass(u[0,0]).cpu().item()))
        energies.append(float(energy(u[0,0]).cpu().item()))

    # error at final step vs truth (simple)
    err = float(F.mse_loss(u, u_truth).cpu().item())
    errs.append(err)

    return {
        "mass": np.array(masses),
        "energy": np.array(energies),
        "final_mse": np.array(errs),
    }


# ----------------------------
# Main: example runs
# ----------------------------

def main():
    set_seed(0)
    grid = Grid1D(nx=128)
    train_cfg = TrainCfg(device="cpu", steps=1500, print_every=300, rollout_steps=200)

    # --- Test A: regime-switch PDE ---
    pde_cfg = RegimePDECfg(dt=0.01, c_adv=1.0, kappa0=0.01, kappa1=0.10)

    # train: exclude regime 1 to test generalization (optional)
    holdout = {"exclude_regime": None}  # set to 1 to train only on r=0

    for mode in ["none", "concat", "film"]:
        model = train_model(
            problem="pde", mode=mode, train_cfg=train_cfg, grid=grid,
            pde_cfg=pde_cfg, holdout=holdout, out_prefix="out"
        )
        # rollout diagnostics for r=0 and r=1
        for r in [0, 1]:
            stats = rollout_pde(model, grid, pde_cfg, mode=mode, r=r, steps=train_cfg.rollout_steps,
                                device=torch.device(train_cfg.device))
            plot_series(stats["mass"], f"PDE rollout mass ({mode}, r={r})", f"out_pde_{mode}_r{r}_mass.png")
            plot_series(stats["energy"], f"PDE rollout energy ({mode}, r={r})", f"out_pde_{mode}_r{r}_energy.png")
            print(f"[pde|{mode}] r={r} final MSE vs truth: {stats['final_mse'][0]:.4e}")

    # --- Test B: season/month toy ---
    season_cfg = SeasonalToyCfg(months=12, noise_std=0.02)
    holdout_months = {"exclude_months": [10, 11]}  # hold out Nov/Dec (0-based: 10,11)
    for mode in ["none", "concat", "film"]:
        _ = train_model(
            problem="season", mode=mode, train_cfg=train_cfg, grid=grid,
            season_cfg=season_cfg, holdout=holdout_months, out_prefix="out"
        )

    # --- Test C: resolution-conditioned closure ---
    closure_cfg = ClosureCfg(dx_choices=(0.5, 1.0, 2.0, 4.0), nu0=0.10, alpha=0.7, noise_std=0.01)
    holdout_scales = {"exclude_scales": [4.0]}  # hold out the coarsest scale
    for mode in ["none", "concat", "film"]:
        _ = train_model(
            problem="closure", mode=mode, train_cfg=train_cfg, grid=grid,
            closure_cfg=closure_cfg, holdout=holdout_scales, out_prefix="out"
        )

    print("Done. Check out_*_gamma/beta.png and rollout mass/energy plots.")

if __name__ == "__main__":
    main()


[pde|none] step   300/1500  loss=1.1696e-03
[pde|none] step   600/1500  loss=2.2071e-04
[pde|none] step   900/1500  loss=1.2105e-04
[pde|none] step  1200/1500  loss=8.4857e-05
[pde|none] step  1500/1500  loss=1.4681e-04
[pde|none] r=0 final MSE vs truth: inf
[pde|none] r=1 final MSE vs truth: inf
[pde|concat] step   300/1500  loss=1.1050e-03
[pde|concat] step   600/1500  loss=1.8978e-04
[pde|concat] step   900/1500  loss=1.3013e-04
[pde|concat] step  1200/1500  loss=6.0501e-05
[pde|concat] step  1500/1500  loss=1.0907e-04
[pde|concat] r=0 final MSE vs truth: inf
[pde|concat] r=1 final MSE vs truth: 1.4855e+27
[pde|film] step   300/1500  loss=2.4167e-03
[pde|film] step   600/1500  loss=7.1774e-05
[pde|film] step   900/1500  loss=3.3408e-05
[pde|film] step  1200/1500  loss=9.1924e-05
[pde|film] step  1500/1500  loss=2.7319e-05
[pde|film] r=0 final MSE vs truth: 6.5828e-01
[pde|film] r=1 final MSE vs truth: 6.5148e-02
[season|none] step   300/1500  loss=1.2775e-01
[season|none] step   600